In [2]:
import cobra
from pathlib import Path
import pandas as pd

model_folder = Path("AGORA2_SBML")

files = sorted(model_folder.glob("*Bifidobacterium_longum*.xml"))

len(files)

for f in files:
    print(f.name)

Bifidobacterium_longum_BG7.xml
Bifidobacterium_longum_BXY01.xml
Bifidobacterium_longum_DJO10A.xml
Bifidobacterium_longum_E18.xml
Bifidobacterium_longum_ERR2221141.xml
Bifidobacterium_longum_ERR2221351.xml
Bifidobacterium_longum_ERR2221409.xml
Bifidobacterium_longum_ERR2230052.xml
Bifidobacterium_longum_ERR2230120.xml
Bifidobacterium_longum_ERR2230133.xml
Bifidobacterium_longum_ERR2230158.xml
Bifidobacterium_longum_infantis_157F_NC.xml
Bifidobacterium_longum_infantis_ATCC_15697.xml
Bifidobacterium_longum_longum_ATCC_55813.xml
Bifidobacterium_longum_longum_BBMN68.xml
Bifidobacterium_longum_longum_CCUG_52486.xml
Bifidobacterium_longum_longum_JCM_1217.xml
Bifidobacterium_longum_longum_JDM301.xml
Bifidobacterium_longum_NCC2705.xml
Bifidobacterium_longum_subsp_infantis_BT1.xml
Bifidobacterium_longum_subsp_longum_1_6B.xml
Bifidobacterium_longum_subsp_longum_2_2B.xml
Bifidobacterium_longum_subsp_longum_35B.xml
Bifidobacterium_longum_subsp_longum_44B.xml
Bifidobacterium_longum_subsp_longum_CMCC

In [3]:
model_folder = Path("AGORA2_SBML")

NCC2705 = model_folder/"Bifidobacterium_longum_NCC2705.xml"

NCC2705_2 = cobra.io.read_sbml_model(NCC2705)

print(NCC2705_2.id)

print(len(NCC2705_2.metabolites))

print(len(NCC2705_2.reactions))

print(len(NCC2705_2.genes))

print(NCC2705_2.compartments)

print(NCC2705_2.objective.expression)

M_Bifidobacterium_longum_NCC2705
926
1008
593
{'c': 'Cytoplasm', 'e': 'Extracellular'}
1.0*biomass525 - 1.0*biomass525_reverse_5c178


In [4]:
exchange_rxns = NCC2705_2.exchanges

exc_num = len(exchange_rxns)

#for rxn in exchange_rxns[:154]:
#    print(rxn.id, rxn.reaction)
    
def_med = NCC2705_2.medium
print(def_med)

max_growth = NCC2705_2.optimize()
print(max_growth.status)
print(NCC2705_2.objective)
print(max_growth.objective_value)

{'EX_12ppd_S(e)': 1000.0, 'EX_4abut(e)': 1000.0, 'EX_4abz(e)': 1000.0, 'EX_4ahmmp(e)': 1000.0, 'EX_5fura(e)': 1000.0, 'EX_5mthf(e)': 1000.0, 'EX_7a_czp(e)': 1000.0, 'EX_C02528(e)': 1000.0, 'EX_HC02191(e)': 1000.0, 'EX_HC02192(e)': 1000.0, 'EX_HC02193(e)': 1000.0, 'EX_M01989(e)': 1000.0, 'EX_M03134(e)': 1000.0, 'EX_ac(e)': 1000.0, 'EX_acald(e)': 1000.0, 'EX_ade(e)': 1000.0, 'EX_adocbl(e)': 1000.0, 'EX_ala_L(e)': 1000.0, 'EX_alaasp(e)': 1000.0, 'EX_alagln(e)': 1000.0, 'EX_alaglu(e)': 1000.0, 'EX_alagly(e)': 1000.0, 'EX_alahis(e)': 1000.0, 'EX_alaleu(e)': 1000.0, 'EX_alathr(e)': 1000.0, 'EX_anzp(e)': 1000.0, 'EX_arab_L(e)': 1000.0, 'EX_arabinogal(e)': 1000.0, 'EX_arabttr(e)': 1000.0, 'EX_asn_L(e)': 1000.0, 'EX_asp_L(e)': 1000.0, 'EX_biomass(e)': 1000.0, 'EX_btn(e)': 1000.0, 'EX_butam(e)': 1000.0, 'EX_ca2(e)': 1000.0, 'EX_cbl1(e)': 1000.0, 'EX_cd2(e)': 1000.0, 'EX_cgly(e)': 1000.0, 'EX_chlphncl(e)': 1000.0, 'EX_cholate(e)': 1000.0, 'EX_cl(e)': 1000.0, 'EX_co2(e)': 1000.0, 'EX_cobalt2(e)': 

In [5]:
model = NCC2705_2
# ---- 1. Close every exchange reaction (block all uptake; still allow secretion) ----
for rxn in model.exchanges:
    rxn.lower_bound = 0
    rxn.upper_bound = 1000

# ---- 2. Bound tiers (placeholders — refine in your sensitivity analysis) ----
CARBON      = 10.0   # glucose
ION_MOD     = 1.0    # phosphate, ammonium, magnesium, sulfate, sodium/acetate
ION_TRACE   = 0.05   # manganese ("expensive", deliberately limiting)
AA_BOUND    = 1.0    # amino acids, standing in for peptone + extracts
VITAMIN_B   = 0.05   # vitamins/cofactors, standing in for extracts
TRACE_META  = 0.01   # Fe/Zn/Co/Cu riding along with extracts
NUCLEOTIDE  = 0.05   # purines/pyrimidines riding along with extracts
FREE        = 1000.0 # water, protons, CO2 — never limiting

def open_rxn(rxn_id, bound):
    if rxn_id in model.reactions:
        model.reactions.get_by_id(rxn_id).lower_bound = -bound

# directly-mapped MRS ingredients
open_rxn("EX_glc_D(e)", CARBON)
open_rxn("EX_pi(e)",    ION_MOD)
open_rxn("EX_nh4(e)",   ION_MOD)
open_rxn("EX_ac(e)",    ION_MOD)
open_rxn("EX_na1(e)",   ION_MOD)
open_rxn("EX_mg2(e)",   ION_MOD)
open_rxn("EX_so4(e)",   ION_MOD)
open_rxn("EX_mn2(e)",   ION_TRACE)

# always-free
for rxn_id in ["EX_h(e)", "EX_h2o(e)", "EX_co2(e)"]:
    open_rxn(rxn_id, FREE)
open_rxn("EX_o2(e)", 0)   # anaerobic default — B. longum is an anaerobe; override if you want aerobic

# peptone + meat extract + yeast extract -> amino acids
amino_acids = ["ala_L","asn_L","asp_L","cys_L","gln_L","glu_L","gly","his_L","ile_L",
               "leu_L","lys_L","met_L","phe_L","pro_L","ser_L","thr_L","trp_L","tyr_L","val_L"]
for aa in amino_acids:
    open_rxn(f"EX_{aa}(e)", AA_BOUND)

# ... -> vitamins/cofactors
vitamins = ["btn","fol","nac","pnto_R","pydam","pydx","pydxn","ribflv","thm","thf","5mthf","cbl1","adocbl","4abz","dpcoa"]
for v in vitamins:
    open_rxn(f"EX_{v}(e)", VITAMIN_B)

# ... -> trace metals and nucleotides
for m in ["fe2","fe3","zn2","cobalt2","cu2"]:
    open_rxn(f"EX_{m}(e)", TRACE_META)
for n in ["ade","gua","hxan","xan","orot"]:
    open_rxn(f"EX_{n}(e)", NUCLEOTIDE)

# ---- 3. Solve ----
solution = model.optimize()
print("Growth rate (MRS, realistic bounds):", solution.objective_value, "h^-1")
print("Status:", solution.status)

Growth rate (MRS, realistic bounds): 1.0454449486337772e-43 h^-1
Status: optimal
